# Solving the Head and Neck Tumor Segmentation for MR-Guided Applications(HNTS-MRG) with UNETR

### Installing and setting up correct dependencies for the environment

In [3]:
!pip install -q "monai-weekly[nibabel, tqdm, einops]"
!python -c "import matplotlib" || pip install -q matplotlib
%matplotlib inline

In [34]:
import os
import shutil
import tempfile
from glob import glob
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from tqdm import tqdm

from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.transforms import (
    AsDiscrete,
    EnsureChannelFirstd,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandFlipd,
    RandCropByPosNegLabeld,
    RandShiftIntensityd,
    ScaleIntensityRanged,
    Spacingd,
    RandRotate90d,
)

from monai.config import print_config
from monai.metrics import DiceMetric
from monai.networks.nets import UNETR

from monai.data import (
    DataLoader,
    CacheDataset,
    load_decathlon_datalist,
    decollate_batch,
)


import torch

print_config()

MONAI version: 1.5.dev2513
Numpy version: 1.26.4
Pytorch version: 2.4.1+cu121
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: ef083a32ccc13ee3937a4bd8acc12b9cdc174e18
MONAI __file__: /home/<username>/.local/lib/python3.10/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.3.2
scikit-image version: 0.21.0
scipy version: 1.8.0
Pillow version: 9.0.1
Tensorboard version: 2.16.2
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.19.1+cu121
tqdm version: 4.66.4
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 5.9.5
pandas version: 1.3.5
einops version: 0.8.1
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing

### Setting up directories for training

Project is set up to access dataset stored at the Cybele lab at NTNU.

In [20]:
directory = "data_directory"
os.makedirs(directory, exist_ok=True)
root_dir = directory

# Define the correct training directory path
base_directory = "/datasets/tdt4265/mic/open/HNTS-MRG"

print(f"Training directory path: {base_directory}")
print(f"Training directory exists: {os.path.exists(base_directory)}")

Training directory path: /datasets/tdt4265/mic/open/HNTS-MRG
Training directory exists: True


### Hyperparameters

In [33]:
train_validation_split = 0.8

In [37]:
# Sort patients into a list in accending order
train_patients = sorted(glob(os.path.join(base_directory, "train", "[0-9]*")), key=lambda x: int(os.path.basename(x)))
test_patients = sorted(glob(os.path.join(base_directory, "test", "[0-9]*")), key=lambda x: int(os.path.basename(x)))

# Organize patients into dictionaries with image and corresponding label for each patient
train_files = []
for patient in train_patients:
    train_files.append({"image": glob(os.path.join(patient, "preRT", "[0-9]*_preRT_T2.nii.gz")), "label": glob(os.path.join(patient, "preRT", "[0-9]*_preRT_mask.nii.gz"))})
test_files = []
for patient in test_patients:
    test_files.append({"image": glob(os.path.join(patient, "preRT", "[0-9]*_preRT_T2.nii.gz")), "label": glob(os.path.join(patient, "preRT", "[0-9]*_preRT_mask.nii.gz"))})
print(test_files)

# Split training into train and validation data
train_files, val_files = train_test_split(train_files, test_size=train_validation_split)

[{'image': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/8/preRT/8_preRT_T2.nii.gz'], 'label': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/8/preRT/8_preRT_mask.nii.gz']}, {'image': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/20/preRT/20_preRT_T2.nii.gz'], 'label': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/20/preRT/20_preRT_mask.nii.gz']}, {'image': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/29/preRT/29_preRT_T2.nii.gz'], 'label': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/29/preRT/29_preRT_mask.nii.gz']}, {'image': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/36/preRT/36_preRT_T2.nii.gz'], 'label': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/36/preRT/36_preRT_mask.nii.gz']}, {'image': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/41/preRT/41_preRT_T2.nii.gz'], 'label': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/41/preRT/41_preRT_mask.nii.gz']}, {'image': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/42/preRT/42_preRT_T2.nii.gz'], 'label': ['/datasets/tdt4265/mic/open/HNTS-MRG/test/42/preRT/42_preRT

### Transforms

Defining transformations for train, test and validation datasets